# Pocket2Mol 引导束搜索 — 端到端可复现演示

本 notebook 证明参赛代码可执行：环境自检 → 模型加载 → 单口袋采样（引导束搜索 λ=3）→ 结果与新颖性核验。

运行前提：`conda activate Pocket2Mol`，工作目录为仓库根，GPU 可用。

In [ ]:
import os, sys, torch
sys.path.insert(0, os.getcwd())
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert os.path.exists('ckpt/pretrained_Pocket2Mol.pt'), '权重缺失'
assert os.path.exists('configs/sample_for_pdb_5010050_baseline_s2024.yml'), '配置缺失'
print('环境自检通过')

## 1. 权重加载（含 PyTorch 2.6 兼容补丁）

In [ ]:
from utils import misc  # 触发 add_safe_globals 补丁
ck = torch.load('ckpt/pretrained_Pocket2Mol.pt', map_location='cpu')
print('参数量:', sum(v.numel() for v in ck['model'].values()), '| keys:', len(ck['model']))

## 2. 引导束搜索采样（A2A 受体 4EIY，约 10 分钟）

配置：50 样本 / 100 束宽 / 50 步，λ=3，diversity_w=0.5。
关闭引导：将 config 中 `sample.guided.enabled` 改为 false 即与原版行为一致。

In [ ]:
!"D:/Miniconda3/envs/Pocket2Mol/python.exe" sample_for_pdb.py --pdb_path "D:/MMModel/靶点结构/4EIY_A2A受体.pdb" --center " -0.4,8.5,17.1" --config configs/sample_for_pdb_5010050_baseline_s2024.yml --outdir outputs/nb_demo

## 3. 结果汇总：QED / SA / 骨架

In [ ]:
import glob, numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import QED, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.*')
f = glob.glob('outputs/nb_demo/*/SMILES.txt')[0]
smis = sorted(set(s for s in open(f, encoding='utf-8').read().splitlines() if s.strip()))
ms = [m for m in (Chem.MolFromSmiles(s) for s in smis) if m]
print('分子数:', len(smis))
print('QED 中位: %.3f' % np.median([QED.qed(m) for m in ms]))
print('MW 中位: %.1f' % np.median([Descriptors.MolWt(m) for m in ms]))
print('唯一骨架:', len({MurckoScaffold.MurckoScaffoldSmiles(mol=m) for m in ms}))

## 4. 后续管线入口
- 过滤入库：`python scripts/build_library.py --runs outputs/nb_demo`
- 对接：`python scripts/docking_pipeline.py --receptor <pdb> --center=... --smiles-file ... --out ...`
- 新颖性：`已知药物库/A2A_known_set.txt`（737 个）对照